In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 3.8 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://customize-watch-lifting.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://customize-watch-lifting.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology, MHUST）是一所位於台灣新竹縣新豐鄉的私立科技大學，以其鄰近新竹科學園區的地理優勢和務實的技職教育聞名。

**簡要介紹：**

1.  **創校與發展：** 學校創立於1966年，前身為「明新工業專科學校」，是當時台灣第一所私立工業專科學校。隨著高等教育體制發展，於1997年升格為「明新技術學院」，並於2002年正式改制為「明新科技大學」。
2.  **地理位置優勢：** 學校位於新竹縣，緊鄰有「台灣矽谷」之稱的新竹科學園區。這使得明新科大在產學合作、學生實習與畢業生就業方面，擁有得天獨厚的條件與豐沛的資源。
3.  **辦學特色：**
    *   **實務導向與產學合作：** 強調「理論與實務並重」，積極與產業鏈結，推動產學合作專案、客製化學程及實務專題製作，確保學生所學能符合業界需求，畢業即能無縫接軌職場。
    *   **高就業率：** 以培育具備專業技能與職場競爭力的實用型人才為目標，畢業生在工程、管理、設計、服務等領域表現良好，深受企業界肯定，享有良好的就業口碑。
    *   **多元學院：** 設有工程學院、管理學院、服務事業學院、設計學院等，涵蓋了工程、資訊、管理、商業、設計、觀光、餐旅等多元專業領域。
    *   **創新與人文素養：** 除了專業技能，也注重培養學生的創新思維、解決問題能力、團隊合作精神與職場倫理，並強調人文素養的培育。

**總結來說，** 明新科技大學是一所致力於提供優質技職教育、與產業緊密結合的科技大學，尤其擅長培育符合產業需求的專業人才，是台灣中部地區重要的應用型高等教育機構之一。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

根據明新科技大學官方網站顯示，目前（截至我最後的知識更新時間）的校長是 **劉國偉** 博士。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt) #當你在 LINE Bot 中連續發送相關問題時（例如先問「簡介明新科技大學」，再問「校長是誰？」），Gemini 模型能夠理解後一個問題是針對前一個問題的上下文，從而給出更精確和相關的回答，而不是將每個問題都視為獨立的查詢。
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ub9ff2c3714925949563d00e62be17749","events":[{"type":"message","message":{"type":"text","id":"615031636513784491","quoteToken":"FlIdRotejC9SYl-HlB_ORjeRjN9YDT826OWObTMWzabiPw_kTxfFgzBFcq7z1HECpo8y_aet_iasVX4EJa7nKPoNoBMzwokJW79PYXFCdhxemYl49kUe45xALL3DRTNakWUnMejTD88F-OzLKjlzRQ","markAsReadToken":"F6TpA6O4gsqwqgG8RgoBHykZyT0gugYmdMtQL3FCFVutCS3PRR5xR9UYw2S2baK_5KrFEIe8NFA5V-_VIq2x9F0oyfucO0mQj_W-kANbmQw4NzpUy7H-gR_njtS4LTaurCZWTnREY0gTRCB-5js76_459H8pqbQZpYGzNfJIuPQtOKPBn3HBxm4NoxZOddTCXRnpe5OWOtn3N1el5xW-FQ","text":"AI 校長是誰"},"webhookEventId":"01KS6SY0AXMRVQWX4D300DP25X","deliveryContext":{"isRedelivery":false},"timestamp":1779418726239,"source":{"type":"user","userId":"U913e7b32d982838f15ac288880e33797"},"replyToken":"a40c2a2d8df44f489695977eca499ee9","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:58:49] "POST / HTTP/1.1" 200 -
